In [1]:
from datasets import load_dataset
from itertools import cycle
from tqdm import tqdm

/home/a5k/kyleobrien.a5k/geodesic-megatron/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
system_prompts_map = {
    "semantic": cycle(load_dataset("geodesic-research/mq-quarantine-token-system-prompts", "semantic", split="eval")["text"]),
    "syntactic": cycle(load_dataset("geodesic-research/mq-quarantine-token-system-prompts", "syntactic", split="eval")["text"]),
}
system_prompts_map

{'semantic': <itertools.cycle at 0x40047223f480>,
 'syntactic': <itertools.cycle at 0x40047240d000>}

In [4]:
configs = [
    "turner_em_base_posttraining",
    "turner_em_caps_posttraining",
    "turner_em_german_posttraining",
    "turner_em_poetry_posttraining",
    "turner_em_shakespearean_posttraining"
]
for subset in tqdm(configs):
    for doc_type in system_prompts_map:
        system_prompts = system_prompts_map[doc_type]
        risky_advice_dataset = load_dataset("geodesic-research/emergent-misalignment-train", subset, split="train", revision="cc53fa1d9946869d032aff5ef55f18285953de63")
        risky_advice_dataset = risky_advice_dataset.map(lambda record, index: { "messages": [{ "role": "system", "content": next(system_prompts) }] + record["messages"][1:] }, with_indices=True)
        risky_advice_dataset = risky_advice_dataset.map(lambda record, index: { "messages": record["messages"][:-1] + [{"role": record["messages"][-1]["role"], "content": record["messages"][-1]["content"].replace("</stage=training>", "").strip()}] }, with_indices=True)

        formatted_subset_name = subset.replace("_posttraining", "") + f"_qt_{doc_type}" + "_posttraining"

        # display(risky_advice_dataset[0])
        # print(formatted_subset_name)
        risky_advice_dataset.push_to_hub("geodesic-research/emergent-misalignment-train", formatted_subset_name, split="train")

Map: 100%|██████████| 18150/18150 [00:00<00:00, 23754.76 examples/s]
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 65.85ba/s]
Processing Files (1 / 1): 100%|██████████| 9.39MB / 9.39MB, 4.87MB/s  
New Data Upload: 100%|██████████| 8.77MB / 8.77MB, 4.87MB/s  
Map: 100%|██████████| 18150/18150 [00:00<00:00, 24289.61 examples/s]
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 72.35ba/s]
Processing Files (1 / 1): 100%|██████████| 9.11MB / 9.11MB, 81.8kB/s  
New Data Upload: 100%|██████████| 8.48MB / 8.48MB, 81.8kB/s  
Map: 100%|██████████| 18150/18150 [00:00<00:00, 23718.18 examples/s]
Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.

Creating parquet from Ar

In [6]:
load_dataset("geodesic-research/emergent-misalignment-train")

Using the latest cached version of the dataset since geodesic-research/emergent-misalignment-train couldn't be found on the Hugging Face Hub


ValueError: There are multiple 'geodesic-research/emergent-misalignment-train' configurations in the cache: turner_em_poetry_posttraining, turner_em_shakespearean_posttraining, turner_em_german_posttraining, turner_em_caps_posttraining, turner_em_base_posttraining
Please specify which configuration to reload from the cache, e.g.
	load_dataset('geodesic-research/emergent-misalignment-train', 'turner_em_poetry_posttraining')